In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from torcheval.metrics.functional import multiclass_auprc
import numpy as np
import torch
from rgnns import GConvGRUModel, GConvLSTMModel, GATGRU_UNI, GATLSTM_UNI, GINLSTM_UNI, GINGRU_UNI, GINEGRU_UNI, GINELSTM_UNI
from models import RGNN_RNN
from rnns import LSTMModel
import torch.nn as nn
import pickle

In [ ]:
def load_obj_pickle(name):
    with open(name + '.pkl', 'rb') as f:
        return pickle.load(f)

def save_obj_pickle(obj, name):
    with open(name + '.pkl', 'wb+') as f:
        pickle.dump(obj, f, pickle.HIGHEST_PROTOCOL)

In [ ]:
train_items = load_obj_pickle("../Data/train_items")
val_items = load_obj_pickle("../Data/val_items")
test_items = load_obj_pickle("../Data/test_items")
class_weights = load_obj_pickle("../Data/class_weights")

In [ ]:
len(train_items), len(val_items), len(test_items)

In [ ]:
def bootstrap_preds_multiclass(probs, true_labels, num_classes=8, num_boot=10000):
    boot_means_auc = np.zeros(num_boot)
    boot_means_auprc = np.zeros(num_boot)
    boot_means_f1 = np.zeros(num_boot)

    np.random.seed(0)
    for i in range(num_boot):
        # Generate indices for resampling
        indices = np.random.choice(range(len(probs)), size=len(probs), replace=True)
        # Resample labels and predictions
        resampled_labels = true_labels[indices]
        resampled_probs = probs[indices]

        # Recalculate AUC, AUPRC, and F1 for resampled data
        resampled_labels_binarized = label_binarize(resampled_labels, classes=np.arange(num_classes))

        try:
            # AUC calculation
            auc_score = roc_auc_score(resampled_labels_binarized, resampled_probs, average='macro', multi_class='ovr')
            boot_means_auc[i] = auc_score

            # AUPRC calculation
            boot_means_auprc[i] = multiclass_auprc(torch.tensor(resampled_probs), torch.tensor(resampled_labels), num_classes=num_classes)

            # F1 calculation
            preds_resampled = resampled_probs.argmax(axis=1)
            _, _, f1_score, _ = precision_recall_fscore_support(resampled_labels, preds_resampled, average='macro', zero_division=0)
            boot_means_f1[i] = f1_score

        except Exception as e:
            print(f"Error in bootstrap iteration {i}: {e}")
            boot_means_auc[i] = 0
            boot_means_auprc[i] = 0
            boot_means_f1[i] = 0

    return boot_means_auc, boot_means_auprc, boot_means_f1

def train(model, loader, criterion, optimizer, h0_n, h0_g, cell_states_0_g):
    model.train()
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for i, batch in enumerate(loader):
        optimizer.zero_grad()

        X = batch["input_graphs"]

        if isinstance(model, RGNN_RNN):
            out, h0_n, h0_g, att_scores, cell_states_0_g = model(X, h0_n, h0_g, cell_states_0_g)
            if isinstance(model, GConvLSTMModel):
                h0_g = [tuple(h.detach() for h in layer) for layer in h0_g]
            else:
                h0_g = [h.detach() for h in h0_g]
        elif isinstance(model, (GATGRU_UNI, GINEGRU_UNI, GINGRU_UNI)):
            out, h0_g, _ = model(X, h0_g)
            h0_g = [h.detach() for h in h0_g]
        elif isinstance(model, (GINLSTM_UNI, GATLSTM_UNI, GINELSTM_UNI)):
            out, h0_g, cell_states_0_g = model(X, (h0_g, cell_states_0_g))
            h0_g = [h.detach() for h in h0_g]
            cell_states_0_g = [c.detach() for c in cell_states_0_g]
        else: # RNN Unimodal
            out, h0_n = model(X, h0_n)

        batch["target_y"] = batch["target_y"].type(torch.LongTensor)
        
        loss = criterion(out[batch["target_mask"]], batch["target_y"][batch["target_mask"]])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()

        total_loss += loss.item()

        correct_predictions += (out[batch["target_mask"]].argmax(dim=1) == batch["target_y"][batch["target_mask"]]).sum().item()
        total_samples += batch["target_y"][batch["target_mask"]].shape[0]

    avg_loss = total_loss / len(loader)
    accuracy = correct_predictions / total_samples

    return avg_loss, accuracy, h0_n, h0_g

def evaluate(model, loader, criterion, h0_n, h0_g, cell_states_0_g, inferencing=False, unseen_indices=[], num_classes=8):
    model.eval()
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    predictions = []
    true_labels = []
    probs = []
    unseen_predictions = []
    unseen_true_labels = []
    unseen_probs = []
    unseen_correct_predictions = 0
    unseen_total_samples = 0

    with torch.no_grad():
        for i, batch in enumerate(loader):
            X = batch["input_graphs"]
            
            if type(model) is RGNN_RNN:
                out, h0_n, h0_g, att_scores, cell_states_0_g = model(X, h0_n, h0_g, cell_states_0_g)
                if type(model.GNN) is GConvLSTMModel:
                    h0_g = [tuple(h.detach() for h in layer) for layer in h0_g]
                else:
                    h0_g = [h.detach() for h in h0_g]
            elif isinstance(model, (GATGRU_UNI, GINEGRU_UNI, GINGRU_UNI)):
                out, h0_g, _ = model(X, h0_g)
                h0_g = [h.detach() for h in h0_g]
            elif isinstance(model, (GINLSTM_UNI, GATLSTM_UNI, GINELSTM_UNI)):
                out, h0_g, cell_states_0_g = model(X, (h0_g, cell_states_0_g))
                h0_g = [h.detach() for h in h0_g]
                cell_states_0_g = [c.detach() for c in cell_states_0_g]
            else: # RNN Unimodal
                out, h0_n = model(X, h0_n)

            batch["target_y"] = batch["target_y"].type(torch.LongTensor)

            loss = criterion(out[batch["target_mask"]], batch["target_y"][batch["target_mask"]])
            total_loss += loss.item()

            y_pred = out[batch["target_mask"]].argmax(dim=1)
            y_true = batch["target_y"][batch["target_mask"]]
            correct_predictions += (y_pred == y_true).sum().item()
            total_samples += batch["target_y"][batch["target_mask"]].shape[0]

            predictions.append(y_pred.cpu().numpy())
            true_labels.append(y_true.cpu().numpy())
            probs.append(out[batch["target_mask"]].cpu().numpy())
            
            # Inference for unseen nodes
            if inferencing:
                # Use unseen_indices to get the unseen nodes
                y_pred_unseen = out[unseen_indices].argmax(dim=1)
                y_true_unseen = batch["target_y"][unseen_indices]
                unseen_correct_predictions += (y_pred_unseen == y_true_unseen).sum().item()
                unseen_total_samples += batch["target_y"][unseen_indices].shape[0]
                unseen_predictions.append(y_pred_unseen.cpu().numpy())
                unseen_true_labels.append(y_true_unseen.cpu().numpy())
                unseen_probs.append(out[unseen_indices].cpu().numpy())


    # Flatten true labels and predictions
    true_labels_flat = np.concatenate(true_labels)
    probs_flat = np.concatenate(probs)
    predictions_flat = np.concatenate(predictions)

    # Binarize true labels for multiclass classification
    true_labels_binarized = label_binarize(true_labels_flat, classes=np.unique(true_labels_flat))

    # Calculate AUC score
    try:
        auc = roc_auc_score(true_labels_binarized, probs_flat, average='macro', multi_class='ovr')
    except ValueError:
        auc = 0

    # Calculate AUPRC score
    try:
        auprc = multiclass_auprc(torch.tensor(probs_flat), torch.tensor(true_labels_flat), num_classes=num_classes)
    except ValueError:
        auprc = 0

    if inferencing:
        unseen_predictions_flat = np.concatenate(unseen_predictions)
        unseen_true_labels_flat = np.concatenate(unseen_true_labels)
        unseen_probs_flat = np.concatenate(unseen_probs)
        accuracy_unseen = unseen_correct_predictions / unseen_total_samples
        precision_unseen, recall_unseen, f1_unseen, _ = precision_recall_fscore_support(unseen_true_labels_flat, unseen_predictions_flat, average='macro', zero_division=0)

        unseen_probs_flat = unseen_probs_flat[:, :-1]
        probs_flat = probs_flat[:, :-1]

        true_labels_binarized_unseen = label_binarize(unseen_true_labels_flat, classes=np.unique(unseen_true_labels_flat))


        auc_unseen = roc_auc_score(true_labels_binarized_unseen, unseen_probs_flat, average='macro', multi_class='ovr')

        num_cls = len(np.unique(unseen_true_labels_flat))
        auprc_unseen = multiclass_auprc(torch.tensor(unseen_probs_flat), torch.tensor(unseen_true_labels_flat), num_classes=num_cls)

        auc_boot_means, auprc_boot_means, f1_boot_means = bootstrap_preds_multiclass(probs_flat, true_labels_flat, num_classes=num_cls, num_boot=10000)
        auc_boot_means_unseen, auprc_boot_means_unseen, f1_boot_means_unseen = bootstrap_preds_multiclass(unseen_probs_flat, unseen_true_labels_flat, num_classes=num_cls, num_boot=10000)
    else:
        auc_boot_means = auprc_boot_means = f1_boot_means = np.zeros(1)
        auc_boot_means_unseen = auprc_boot_means_unseen = f1_boot_means_unseen = np.zeros(1)
        accuracy_unseen = precision_unseen = recall_unseen = f1_unseen = auc_unseen = auprc_unseen = 0


    avg_loss = total_loss / len(loader)
    accuracy = correct_predictions / total_samples
    unseen_accuracy = unseen_correct_predictions / unseen_total_samples if inferencing and unseen_total_samples > 0 else 0
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels_flat, predictions_flat, average='macro', zero_division=0)

    return avg_loss, accuracy, precision, recall, f1, auc, auprc, auc_boot_means, auprc_boot_means, f1_boot_means, accuracy_unseen, precision_unseen, recall_unseen, f1_unseen, auc_unseen, auprc_unseen, auc_boot_means_unseen, auprc_boot_means_unseen, f1_boot_means_unseen, unseen_accuracy

In [ ]:
train_items[0]

In [ ]:
# Node count

train_items[0]["input_graphs"][0].x.shape[0]

In [ ]:
# Numerical features count

train_items[0]["input_graphs"][0].x.shape[1]

In [ ]:
class_weights

In [ ]:
class_weights = [class_weights[rating] for rating in sorted(class_weights)]
class_weights = torch.tensor(class_weights, dtype=torch.float32)
class_weights

In [ ]:
train_items[0]["input_graphs"][0].x.shape

In [ ]:
# Model parameters
in_channels = train_items[0]["input_graphs"][0].x.shape[1]
rnn_hidden_channels = 128
gnn_hidden_channels = 128
num_classes = 8
num_heads = 4
num_gnn_layers = 4
num_rnn_layers = 4
edge_dim = 1
num_nodes = train_items[0]["input_graphs"][0].x.shape[0] # 1402
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)

# Multi-modal RGNN-RNN model
model = RGNN_RNN(num_features=in_channels, rnn_hidden_dim=rnn_hidden_channels, gnn_hidden_dim=gnn_hidden_channels,  num_classes=num_classes, num_gnn_layers=num_gnn_layers, num_rnn_layers=num_rnn_layers, edge_dim=edge_dim, num_heads=num_heads, num_nodes=num_nodes, gnn_model=GConvGRUModel, rnn_model=LSTMModel).to(device)

# Uni-modal RNN model
# model = LSTMModel(input_dim=in_channels, hidden_dim=rnn_hidden_channels, n_layers=num_rnn_layers, n_nodes=num_nodes).to(device)

# Uni-modal RGNN model
# model = GINLSTM_UNI(in_channels=in_channels, out_channels=gnn_hidden_channels, num_layers=num_gnn_layers, n_nodes=num_nodes).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4,
    betas=(0.9, 0.999),
    eps=1e-8
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)


In [ ]:
# rnn hidden states, gnn hidden states, gnn cell states

h0_n, h0_g, cell_states_0_g = None, None, None

In [ ]:
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
aucs = []
aucprcs = []
val_unseen_aucs = []
val_unseen_aucprcs = []

In [ ]:
# Training loop
epochs = 60

for epoch in range(epochs):
    train_loss, train_accuracy, h0_n, h0_g = train(model, train_items, criterion, optimizer, h0_n, h0_g, cell_states_0_g)
    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    print(f'Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, LR: {optimizer.param_groups[0]["lr"]:.6f}')

    val_loss, val_accuracy, precision, recall, f1, auc, auprc, _, _, _, _, _, _, _, _, _, _, _, _, _ = evaluate(model, val_items, criterion, h0_n, h0_g, cell_states_0_g, False)
    scheduler.step(val_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)
    aucs.append(auc)
    aucprcs.append(auprc)
    print(f'Epoch {epoch + 1}/{epochs} - Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, AUPRC: {auprc:.4f}')



In [ ]:
# Without unseen classification

test_loss, test_accuracy, precision, recall, f1, auc, auprc, auc_boot_means, auprc_boot_means, f1_boot_means, accuracy_unseen, precision_unseen, recall_unseen, f1_unseen, auc_unseen, auprc_unseen, auc_boot_means_unseen, auprc_boot_means_unseen, f1_boot_means_unseen, unseen_accuracy = evaluate(model, test_items, criterion, h0_n, h0_g, cell_states_0_g, False, unseen_indices)

print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, AUPRC: {auprc:.4f}, Accuracy Unseen: {accuracy_unseen:.4f}, Precision Unseen: {precision_unseen:.4f}, Recall Unseen: {recall_unseen:.4f}, F1 Unseen: {f1_unseen:.4f}, AUC Unseen: {auc_unseen:.4f}, AUPRC Unseen: {auprc_unseen:.4f}')
print(f"Mean AUC: {auc_boot_means.mean():.4f}, Mean AUPRC: {auprc_boot_means.mean():.4f}, Mean F1: {f1_boot_means.mean():.4f}, Mean AUC Unseen: {auc_boot_means_unseen.mean():.4f}, Mean AUPRC Unseen: {auprc_boot_means_unseen.mean():.4f}, Mean F1 Unseen: {f1_boot_means_unseen.mean():.4f}")

In [ ]:
# With unseen classification

test_loss, test_accuracy, precision, recall, f1, auc, auprc, auc_boot_means, auprc_boot_means, f1_boot_means, accuracy_unseen, precision_unseen, recall_unseen, f1_unseen, auc_unseen, auprc_unseen, auc_boot_means_unseen, auprc_boot_means_unseen, f1_boot_means_unseen, unseen_accuracy = evaluate(model, test_items, criterion, h0_n, h0_g, cell_states_0_g, True, unseen_indices)
print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, AUPRC: {auprc:.4f}, Accuracy Unseen: {accuracy_unseen:.4f}, Precision Unseen: {precision_unseen:.4f}, Recall Unseen: {recall_unseen:.4f}, F1 Unseen: {f1_unseen:.4f}, AUC Unseen: {auc_unseen:.4f}, AUPRC Unseen: {auprc_unseen:.4f}')
print(f"Mean AUC: {auc_boot_means.mean():.4f}, Mean AUPRC: {auprc_boot_means.mean():.4f}, Mean F1: {f1_boot_means.mean():.4f}, Mean AUC Unseen: {auc_boot_means_unseen.mean():.4f}, Mean AUPRC Unseen: {auprc_boot_means_unseen.mean():.4f}, Mean F1 Unseen: {f1_boot_means_unseen.mean():.4f}")

In [ ]:
# Get the auc results with 95% confidence interval
auc_lower = np.percentile(auc_boot_means, 2.5)
auc_upper = np.percentile(auc_boot_means, 97.5)
auc_mean = auc_boot_means.mean()
auc_mean_dist_lower = auc_mean - auc_lower
auc_mean_dist_upper = auc_upper - auc_mean

# Get the auprc results with 95% confidence interval
auprc_lower = np.percentile(auprc_boot_means, 2.5)
auprc_upper = np.percentile(auprc_boot_means, 97.5)
auprc_mean = auprc_boot_means.mean()
auprc_mean_dist_lower = auprc_mean - auprc_lower
auprc_mean_dist_upper = auprc_upper - auprc_mean

# Get the f1 results with 95% confidence interval
f1_lower = np.percentile(f1_boot_means, 2.5)
f1_upper = np.percentile(f1_boot_means, 97.5)
f1_mean = f1_boot_means.mean()
f1_mean_dist_lower = f1_mean - f1_lower
f1_mean_dist_upper = f1_upper - f1_mean

# Get the auc results with 95% confidence interval for unseen nodes
auc_unseen_lower = np.percentile(auc_boot_means_unseen, 2.5)
auc_unseen_upper = np.percentile(auc_boot_means_unseen, 97.5)
auc_unseen_mean = auc_boot_means_unseen.mean()
auc_unseen_mean_dist_lower = auc_unseen_mean - auc_unseen_lower
auc_unseen_mean_dist_upper = auc_unseen_upper - auc_unseen_mean

# Get the auprc results with 95% confidence interval for unseen nodes
auprc_unseen_lower = np.percentile(auprc_boot_means_unseen, 2.5)
auprc_unseen_upper = np.percentile(auprc_boot_means_unseen, 97.5)
auprc_unseen_mean = auprc_boot_means_unseen.mean()
auprc_unseen_mean_dist_lower = auprc_unseen_mean - auprc_unseen_lower
auprc_unseen_mean_dist_upper = auprc_unseen_upper - auprc_unseen_mean

# Get the f1 results with 95% confidence interval for unseen nodes
f1_unseen_lower = np.percentile(f1_boot_means_unseen, 2.5)
f1_unseen_upper = np.percentile(f1_boot_means_unseen, 97.5)
f1_unseen_mean = f1_boot_means_unseen.mean()
f1_unseen_mean_dist_lower = f1_unseen_mean - f1_unseen_lower
f1_unseen_mean_dist_upper = f1_unseen_upper - f1_unseen_mean

print(f"AUC: {auc_mean:.4f} ({auc_mean_dist_lower:.4f}, {auc_mean_dist_upper:.4f})")
print(f"AUPRC: {auprc_mean:.4f} ({auprc_mean_dist_lower:.4f}, {auprc_mean_dist_upper:.4f})")
print(f"F1: {f1_mean:.4f} ({f1_mean_dist_lower:.4f}, {f1_mean_dist_upper:.4f})")
print(f"AUC Unseen: {auc_unseen_mean:.4f} ({auc_unseen_mean_dist_lower:.4f}, {auc_unseen_mean_dist_upper:.4f})")
print(f"AUPRC Unseen: {auprc_unseen_mean:.4f} ({auprc_unseen_mean_dist_lower:.4f}, {auprc_unseen_mean_dist_upper:.4f})")
print(f"F1 Unseen: {f1_unseen_mean:.4f} ({f1_unseen_mean_dist_lower:.4f}, {f1_unseen_mean_dist_upper:.4f})")